# 💾 Veri Kaydetme ve İşleme: CSV, JSON, Database

**Scraping yaptıktan sonra veriyi nasıl saklarız?**

## 🎯 Bu Derste Öğrenecekleriniz:
1. **CSV Formatında** kaydetme
2. **JSON Formatında** kaydetme  
3. **SQLite Database** kullanımı
4. **Pandas ile** veri işleme
5. **Data Cleaning** - Veri temizleme
6. **Incremental Updates** - Artırımlı güncellemeler
7. **File Organization** - Dosya organizasyonu
8. **Data Validation** - Veri doğrulama

---

## 🤔 Neden Veri Kaydetmek Önemli?

### 📊 Veri Kalıcılığı
- **Scraping sonuçları** kaybolmaz
- **Tekrar analiz** yapabilirsiniz
- **Paylaşım** kolaylaşır
- **Backup** oluşturabilirsiniz

### 🔄 Veri Pipeline'ı
```
Web Scraping → Data Processing → Storage → Analysis → Insights
```

### 📈 Format Seçimi
- **CSV**: Excel, Google Sheets uyumlu
- **JSON**: Web API'ler, NoSQL uyumlu
- **SQLite**: Relational data, queries
- **Pandas**: Data science, analysis

In [1]:
# Gerekli kütüphaneleri yükleyelim
import csv
import json
import sqlite3
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
from pathlib import Path
import shutil

print("✅ Temel kütüphaneler yüklendi!")

# Pandas versiyonunu kontrol et
try:
    print(f"📊 Pandas versiyonu: {pd.__version__}")
except:
    print("⚠️ Pandas yüklü değil. Yüklemek için: pip install pandas")

print("\n💾 Veri kaydetme dersine hazırız!")

✅ Temel kütüphaneler yüklendi!
📊 Pandas versiyonu: 2.1.0

💾 Veri kaydetme dersine hazırız!


---

## 🕷️ Örnek 1: Veri Toplama (Hızlı Review)

**Önce veri toplayalım, sonra kaydetme yöntemlerini öğrenelim**

Quotes to Scrape sitesinden veri çekerek başlayalım:

In [2]:
def scrape_quotes_data(max_pages=3):
    """
    Quotes to Scrape sitesinden veri çeker
    Kaydetme örnekleri için sample data oluşturur
    """
    
    print(f"🕷️ VERİ TOPLAMA BAŞLADI")
    print(f"📄 Maksimum sayfa: {max_pages}")
    print("=" * 50)
    
    base_url = "http://quotes.toscrape.com"
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Educational Bot; Data Collection)'
    })
    
    all_quotes = []
    
    for page_num in range(1, max_pages + 1):
        page_url = f"{base_url}?page={page_num}"
        print(f"\n📄 Sayfa {page_num}: {page_url}")
        
        try:
            response = session.get(page_url)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                quotes = soup.find_all('div', class_='quote')
                
                page_data = []
                for quote in quotes:
                    # Veri çıkarma
                    text_elem = quote.find('span', class_='text')
                    author_elem = quote.find('small', class_='author')
                    tags_elems = quote.find_all('a', class_='tag')
                    
                    if text_elem and author_elem:
                        quote_data = {
                            'id': len(all_quotes) + len(page_data) + 1,  # Unique ID
                            'text': text_elem.get_text(strip=True),
                            'author': author_elem.get_text(strip=True),
                            'tags': [tag.get_text(strip=True) for tag in tags_elems],
                            'tags_str': ', '.join([tag.get_text(strip=True) for tag in tags_elems]),  # CSV için
                            'word_count': len(text_elem.get_text(strip=True).split()),
                            'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                            'page_number': page_num
                        }
                        page_data.append(quote_data)
                
                all_quotes.extend(page_data)
                print(f"   ✅ {len(page_data)} alıntı toplandı (Toplam: {len(all_quotes)})")
                
                if not page_data:  # Boş sayfa = pagination sonu
                    print(f"   🏁 Boş sayfa - pagination sonu")
                    break
            
            else:
                print(f"   ❌ HTTP hatası: {response.status_code}")
                break
        
        except Exception as e:
            print(f"   ❌ Hata: {e}")
            break
        
        # Rate limiting
        time.sleep(1)
    
    print(f"\n📊 VERİ TOPLAMA TAMAMLANDI")
    print(f"   📚 Toplam alıntı: {len(all_quotes)}")
    print(f"   📄 İşlenen sayfa: {min(page_num, max_pages)}")
    
    return all_quotes

# Test verisi toplayalım
sample_data = scrape_quotes_data(max_pages=3)

# İlk birkaç kayıt görüntüle
if sample_data:
    print(f"\n🔍 İLK 2 KAYIT ÖRNEĞİ:")
    for i, quote in enumerate(sample_data[:2], 1):
        print(f"\n{i}. ID: {quote['id']}")
        print(f"   Text: {quote['text'][:50]}...")
        print(f"   Author: {quote['author']}")
        print(f"   Tags: {quote['tags_str']}")
        print(f"   Words: {quote['word_count']}")
        print(f"   Scraped: {quote['scraped_at']}")

print("\n✅ Örnek veri hazırlandı! Şimdi kaydetme yöntemlerini öğrenelim.")

🕷️ VERİ TOPLAMA BAŞLADI
📄 Maksimum sayfa: 3

📄 Sayfa 1: http://quotes.toscrape.com?page=1
   ✅ 10 alıntı toplandı (Toplam: 10)

📄 Sayfa 2: http://quotes.toscrape.com?page=2
   ✅ 10 alıntı toplandı (Toplam: 20)

📄 Sayfa 3: http://quotes.toscrape.com?page=3
   ✅ 10 alıntı toplandı (Toplam: 30)

📊 VERİ TOPLAMA TAMAMLANDI
   📚 Toplam alıntı: 30
   📄 İşlenen sayfa: 3

🔍 İLK 2 KAYIT ÖRNEĞİ:

1. ID: 1
   Text: “The world as we have created it is a process of o...
   Author: Albert Einstein
   Tags: change, deep-thoughts, thinking, world
   Words: 21
   Scraped: 2025-08-18 17:53:21

2. ID: 2
   Text: “It is our choices, Harry, that show what we truly...
   Author: J.K. Rowling
   Tags: abilities, choices
   Words: 16
   Scraped: 2025-08-18 17:53:21

✅ Örnek veri hazırlandı! Şimdi kaydetme yöntemlerini öğrenelim.


---

## 📄 Örnek 2: CSV Formatında Kaydetme

**En yaygın format: Excel ve Google Sheets uyumlu**

### 🎯 CSV'nin Avantajları:
- **Excel** ile açılabilir
- **Google Sheets** import edebilir
- **Hafif** dosya boyutu
- **Standart** format
- **İnsan tarafından** okunabilir

In [3]:
def save_to_csv_basic(data, filename='quotes_basic.csv'):
    """
    Temel CSV kaydetme yöntemi
    """
    
    print(f"📄 TEMEL CSV KAYDETME")
    print(f"📁 Dosya: {filename}")
    print("=" * 40)
    
    if not data:
        print("❌ Kaydedilecek veri yok!")
        return
    
    try:
        # CSV writer ile kaydetme
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            # Header (sütun isimleri) belirle
            fieldnames = ['id', 'text', 'author', 'tags_str', 'word_count', 'scraped_at', 'page_number']
            
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            # Header yaz
            writer.writeheader()
            
            # Veriyi yaz
            for item in data:
                # CSV için sadece gerekli alanları al
                csv_row = {field: item.get(field, '') for field in fieldnames}
                writer.writerow(csv_row)
        
        # Dosya bilgilerini göster
        file_size = os.path.getsize(filename)
        print(f"✅ CSV dosyası oluşturuldu!")
        print(f"   📊 Kayıt sayısı: {len(data)}")
        print(f"   📁 Dosya boyutu: {file_size:,} bytes")
        print(f"   🔗 Dosya yolu: {os.path.abspath(filename)}")
        
        # İlk birkaç satırı okuyup göster
        print(f"\n🔍 Dosya içeriği (ilk 3 satır):")
        with open(filename, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i < 3:
                    print(f"   {i+1}. {line.strip()[:80]}...")
                else:
                    break
        
    except Exception as e:
        print(f"❌ CSV kaydetme hatası: {e}")

# Temel CSV kaydetme test et
save_to_csv_basic(sample_data, 'quotes_basic.csv')

📄 TEMEL CSV KAYDETME
📁 Dosya: quotes_basic.csv
✅ CSV dosyası oluşturuldu!
   📊 Kayıt sayısı: 30
   📁 Dosya boyutu: 5,177 bytes
   🔗 Dosya yolu: c:\Users\iamem\Desktop\Web-Scraping-Journey\02-bs4-requests\quotes_basic.csv

🔍 Dosya içeriği (ilk 3 satır):
   1. id,text,author,tags_str,word_count,scraped_at,page_number...
   2. 1,“The world as we have created it is a process of our thinking. It cannot be ch...
   3. 2,"“It is our choices, Harry, that show what we truly are, far more than our abi...


In [4]:
def save_to_csv_advanced(data, filename='quotes_advanced.csv'):
    """
    Gelişmiş CSV kaydetme - özelleştirmeler ile
    """
    
    print(f"📄 GELİŞMİŞ CSV KAYDETME")
    print(f"📁 Dosya: {filename}")
    print("=" * 40)
    
    if not data:
        print("❌ Kaydedilecek veri yok!")
        return
    
    try:
        # Klasör oluştur (eğer yoksa)
        output_dir = 'scraped_data'
        os.makedirs(output_dir, exist_ok=True)
        
        filepath = os.path.join(output_dir, filename)
        
        # Gelişmiş CSV ayarları
        with open(filepath, 'w', newline='', encoding='utf-8') as csvfile:
            # Özel delimiter ve quoting
            fieldnames = [
                'quote_id', 'quote_text', 'author_name', 
                'categories', 'word_count', 'character_count',
                'collection_date', 'source_page'
            ]
            
            writer = csv.DictWriter(
                csvfile, 
                fieldnames=fieldnames,
                delimiter=',',  # Virgül ayıracı
                quotechar='"',  # Tırnak karakteri
                quoting=csv.QUOTE_MINIMAL  # Minimal quoting
            )
            
            # Header yaz
            writer.writeheader()
            
            # Veriyi işleyerek yaz
            for item in data:
                # Veri temizleme ve dönüştürme
                cleaned_text = item['text'].replace('\n', ' ').replace('\r', ' ').strip()
                
                csv_row = {
                    'quote_id': f"Q{item['id']:04d}",  # Q0001 formatında ID
                    'quote_text': cleaned_text,
                    'author_name': item['author'].title(),  # İlk harfleri büyük
                    'categories': item['tags_str'],
                    'word_count': item['word_count'],
                    'character_count': len(cleaned_text),
                    'collection_date': item['scraped_at'],
                    'source_page': f"Page {item['page_number']}"
                }
                
                writer.writerow(csv_row)
        
        # Dosya istatistikleri
        file_size = os.path.getsize(filepath)
        
        print(f"✅ Gelişmiş CSV dosyası oluşturuldu!")
        print(f"   📊 Kayıt sayısı: {len(data)}")
        print(f"   📁 Dosya boyutu: {file_size:,} bytes")
        print(f"   📂 Klasör: {output_dir}/")
        print(f"   🔗 Tam yol: {os.path.abspath(filepath)}")
        
        # Veri özeti
        authors = set(item['author'] for item in data)
        avg_words = sum(item['word_count'] for item in data) / len(data)
        
        print(f"\n📈 VERİ ÖZETİ:")
        print(f"   ✏️ Farklı yazar sayısı: {len(authors)}")
        print(f"   📝 Ortalama kelime sayısı: {avg_words:.1f}")
        print(f"   📄 Toplam sayfa sayısı: {max(item['page_number'] for item in data)}")
        
    except Exception as e:
        print(f"❌ Gelişmiş CSV kaydetme hatası: {e}")

# Gelişmiş CSV kaydetme test et
save_to_csv_advanced(sample_data, 'quotes_advanced.csv')

📄 GELİŞMİŞ CSV KAYDETME
📁 Dosya: quotes_advanced.csv
✅ Gelişmiş CSV dosyası oluşturuldu!
   📊 Kayıt sayısı: 30
   📁 Dosya boyutu: 5,568 bytes
   📂 Klasör: scraped_data/
   🔗 Tam yol: c:\Users\iamem\Desktop\Web-Scraping-Journey\02-bs4-requests\scraped_data\quotes_advanced.csv

📈 VERİ ÖZETİ:
   ✏️ Farklı yazar sayısı: 8
   📝 Ortalama kelime sayısı: 17.1
   📄 Toplam sayfa sayısı: 3


---

## 🌐 Örnek 3: JSON Formatında Kaydetme

**Web API'leri ve modern uygulamalar için ideal**

### 🎯 JSON'un Avantajları:
- **Hierarchical data** - İç içe veri yapıları
- **JavaScript** uyumlu
- **REST API** standardı
- **NoSQL** database uyumlu
- **Metadata** ekleyebilme

In [5]:
def save_to_json_basic(data, filename='quotes_basic.json'):
    """
    Temel JSON kaydetme yöntemi
    """
    
    print(f"🌐 TEMEL JSON KAYDETME")
    print(f"📁 Dosya: {filename}")
    print("=" * 40)
    
    if not data:
        print("❌ Kaydedilecek veri yok!")
        return
    
    try:
        # Basit JSON kaydetme
        with open(filename, 'w', encoding='utf-8') as jsonfile:
            json.dump(
                data, 
                jsonfile, 
                ensure_ascii=False,  # Türkçe karakterler için
                indent=2,  # Okunabilir format
                sort_keys=True  # Anahtarları sırala
            )
        
        # Dosya bilgileri
        file_size = os.path.getsize(filename)
        
        print(f"✅ JSON dosyası oluşturuldu!")
        print(f"   📊 Kayıt sayısı: {len(data)}")
        print(f"   📁 Dosya boyutu: {file_size:,} bytes")
        print(f"   🔗 Dosya yolu: {os.path.abspath(filename)}")
        
        # JSON doğrulama - tekrar oku
        with open(filename, 'r', encoding='utf-8') as jsonfile:
            loaded_data = json.load(jsonfile)
            print(f"   ✅ JSON doğrulaması başarılı: {len(loaded_data)} kayıt")
        
    except Exception as e:
        print(f"❌ JSON kaydetme hatası: {e}")

def save_to_json_structured(data, filename='quotes_structured.json'):
    """
    Yapılandırılmış JSON kaydetme - metadata ile
    """
    
    print(f"\n🌐 YAPILANDIRILMIŞ JSON KAYDETME")
    print(f"📁 Dosya: {filename}")
    print("=" * 40)
    
    if not data:
        print("❌ Kaydedilecek veri yok!")
        return
    
    try:
        # Klasör oluştur
        output_dir = 'scraped_data'
        os.makedirs(output_dir, exist_ok=True)
        filepath = os.path.join(output_dir, filename)
        
        # Veri analizi
        authors = {}
        tags = {}
        
        for item in data:
            # Yazar istatistikleri
            author = item['author']
            if author not in authors:
                authors[author] = {'count': 0, 'total_words': 0}
            authors[author]['count'] += 1
            authors[author]['total_words'] += item['word_count']
            
            # Tag istatistikleri
            for tag in item['tags']:
                tags[tag] = tags.get(tag, 0) + 1
        
        # Yapılandırılmış JSON oluştur
        structured_data = {
            "metadata": {
                "collection_info": {
                    "source": "quotes.toscrape.com",
                    "collected_at": datetime.now().isoformat(),
                    "total_quotes": len(data),
                    "unique_authors": len(authors),
                    "unique_tags": len(tags),
                    "scraping_method": "requests + BeautifulSoup"
                },
                "statistics": {
                    "avg_words_per_quote": sum(item['word_count'] for item in data) / len(data),
                    "max_words": max(item['word_count'] for item in data),
                    "min_words": min(item['word_count'] for item in data),
                    "pages_scraped": max(item['page_number'] for item in data)
                },
                "top_authors": sorted(
                    [(author, stats['count']) for author, stats in authors.items()], 
                    key=lambda x: x[1], reverse=True
                )[:5],
                "top_tags": sorted(tags.items(), key=lambda x: x[1], reverse=True)[:10]
            },
            "quotes": data
        }
        
        # JSON dosyası yaz
        with open(filepath, 'w', encoding='utf-8') as jsonfile:
            json.dump(
                structured_data, 
                jsonfile, 
                ensure_ascii=False,
                indent=2,
                sort_keys=False  # Metadata önce gelsin
            )
        
        # Sonuçları göster
        file_size = os.path.getsize(filepath)
        
        print(f"✅ Yapılandırılmış JSON oluşturuldu!")
        print(f"   📊 Ana veri: {len(data)} alıntı")
        print(f"   📈 Metadata: İstatistikler dahil")
        print(f"   📁 Dosya boyutu: {file_size:,} bytes")
        print(f"   🔗 Tam yol: {os.path.abspath(filepath)}")
        
        # Metadata özetini göster
        metadata = structured_data['metadata']
        print(f"\n📊 METADATA ÖZETİ:")
        print(f"   🏆 En aktif yazar: {metadata['top_authors'][0][0]} ({metadata['top_authors'][0][1]} alıntı)")
        print(f"   🏷️ En popüler tag: {metadata['top_tags'][0][0]} ({metadata['top_tags'][0][1]} kez)")
        print(f"   📝 Ortalama kelime: {metadata['statistics']['avg_words_per_quote']:.1f}")
        
    except Exception as e:
        print(f"❌ Yapılandırılmış JSON kaydetme hatası: {e}")

# JSON kaydetme testleri
save_to_json_basic(sample_data, 'quotes_basic.json')
save_to_json_structured(sample_data, 'quotes_structured.json')

🌐 TEMEL JSON KAYDETME
📁 Dosya: quotes_basic.json
✅ JSON dosyası oluşturuldu!
   📊 Kayıt sayısı: 30
   📁 Dosya boyutu: 11,574 bytes
   🔗 Dosya yolu: c:\Users\iamem\Desktop\Web-Scraping-Journey\02-bs4-requests\quotes_basic.json
   ✅ JSON doğrulaması başarılı: 30 kayıt

🌐 YAPILANDIRILMIŞ JSON KAYDETME
📁 Dosya: quotes_structured.json
✅ Yapılandırılmış JSON oluşturuldu!
   📊 Ana veri: 30 alıntı
   📈 Metadata: İstatistikler dahil
   📁 Dosya boyutu: 13,692 bytes
   🔗 Tam yol: c:\Users\iamem\Desktop\Web-Scraping-Journey\02-bs4-requests\scraped_data\quotes_structured.json

📊 METADATA ÖZETİ:
   🏆 En aktif yazar: Albert Einstein (9 alıntı)
   🏷️ En popüler tag: inspirational (9 kez)
   📝 Ortalama kelime: 17.1


---

## 🗃️ Örnek 4: SQLite Database Kullanımı

**Relational data ve karmaşık sorgular için ideal**

### 🎯 SQLite'in Avantajları:
- **SQL sorguları** çalıştırabilme
- **İlişkisel veri** yapısı
- **ACID** özellikleri
- **Dosya bazlı** - kolay paylaşım
- **Büyük veri setleri** için performanslı

In [6]:
def save_to_sqlite(data, db_filename='quotes.db'):
    """
    SQLite database'e veri kaydetme
    """
    
    print(f"🗃️ SQLITE DATABASE KAYDETME")
    print(f"📁 Database: {db_filename}")
    print("=" * 40)
    
    if not data:
        print("❌ Kaydedilecek veri yok!")
        return
    
    try:
        # Klasör oluştur
        output_dir = 'scraped_data'
        os.makedirs(output_dir, exist_ok=True)
        db_path = os.path.join(output_dir, db_filename)
        
        # SQLite bağlantısı
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Tablolar oluştur
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS quotes (
                id INTEGER PRIMARY KEY,
                text TEXT NOT NULL,
                author_id INTEGER,
                word_count INTEGER,
                scraped_at TIMESTAMP,
                page_number INTEGER,
                FOREIGN KEY (author_id) REFERENCES authors (id)
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS authors (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE NOT NULL,
                quote_count INTEGER DEFAULT 0
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS tags (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT UNIQUE NOT NULL
            )
        ''')
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS quote_tags (
                quote_id INTEGER,
                tag_id INTEGER,
                PRIMARY KEY (quote_id, tag_id),
                FOREIGN KEY (quote_id) REFERENCES quotes (id),
                FOREIGN KEY (tag_id) REFERENCES tags (id)
            )
        ''')
        
        print("✅ Database tabloları oluşturuldu")
        
        # Veri ekleme
        for item in data:
            # Yazar ekle/güncelle
            cursor.execute(
                "INSERT OR IGNORE INTO authors (name) VALUES (?)",
                (item['author'],)
            )
            
            cursor.execute(
                "SELECT id FROM authors WHERE name = ?",
                (item['author'],)
            )
            author_id = cursor.fetchone()[0]
            
            # Quote ekle
            cursor.execute('''
                INSERT OR REPLACE INTO quotes 
                (id, text, author_id, word_count, scraped_at, page_number)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (
                item['id'],
                item['text'],
                author_id,
                item['word_count'],
                item['scraped_at'],
                item['page_number']
            ))
            
            # Tags ekle
            for tag_name in item['tags']:
                cursor.execute(
                    "INSERT OR IGNORE INTO tags (name) VALUES (?)",
                    (tag_name,)
                )
                
                cursor.execute(
                    "SELECT id FROM tags WHERE name = ?",
                    (tag_name,)
                )
                tag_id = cursor.fetchone()[0]
                
                cursor.execute(
                    "INSERT OR IGNORE INTO quote_tags (quote_id, tag_id) VALUES (?, ?)",
                    (item['id'], tag_id)
                )
        
        # Yazar istatistiklerini güncelle
        cursor.execute('''
            UPDATE authors SET quote_count = (
                SELECT COUNT(*) FROM quotes WHERE author_id = authors.id
            )
        ''')
        
        conn.commit()
        
        # İstatistikleri göster
        cursor.execute("SELECT COUNT(*) FROM quotes")
        total_quotes = cursor.fetchone()[0]
        
        cursor.execute("SELECT COUNT(*) FROM authors")
        total_authors = cursor.fetchone()[0]
        
        cursor.execute("SELECT COUNT(*) FROM tags")
        total_tags = cursor.fetchone()[0]
        
        file_size = os.path.getsize(db_path)
        
        print(f"\n✅ SQLite database oluşturuldu!")
        print(f"   📊 Toplam alıntı: {total_quotes}")
        print(f"   ✏️ Toplam yazar: {total_authors}")
        print(f"   🏷️ Toplam tag: {total_tags}")
        print(f"   📁 Dosya boyutu: {file_size:,} bytes")
        print(f"   🔗 Database yolu: {os.path.abspath(db_path)}")
        
        # Örnek sorgular
        print(f"\n🔍 ÖRNEK SORGULAR:")
        
        # En aktif yazarlar
        cursor.execute(
            "SELECT name, quote_count FROM authors ORDER BY quote_count DESC LIMIT 3"
        )
        top_authors = cursor.fetchall()
        print(f"   🏆 En aktif yazarlar:")
        for name, count in top_authors:
            print(f"      • {name}: {count} alıntı")
        
        # En uzun alıntı
        cursor.execute(
            "SELECT text, word_count FROM quotes ORDER BY word_count DESC LIMIT 1"
        )
        longest_quote = cursor.fetchone()
        if longest_quote:
            print(f"   📝 En uzun alıntı: {longest_quote[1]} kelime")
            print(f"      {longest_quote[0][:50]}...")
        
        conn.close()
        
    except Exception as e:
        print(f"❌ SQLite kaydetme hatası: {e}")
        if 'conn' in locals():
            conn.close()

# SQLite test et
save_to_sqlite(sample_data, 'quotes.db')

🗃️ SQLITE DATABASE KAYDETME
📁 Database: quotes.db
✅ Database tabloları oluşturuldu

✅ SQLite database oluşturuldu!
   📊 Toplam alıntı: 30
   ✏️ Toplam yazar: 8
   🏷️ Toplam tag: 26
   📁 Dosya boyutu: 36,864 bytes
   🔗 Database yolu: c:\Users\iamem\Desktop\Web-Scraping-Journey\02-bs4-requests\scraped_data\quotes.db

🔍 ÖRNEK SORGULAR:
   🏆 En aktif yazarlar:
      • Albert Einstein: 9 alıntı
      • J.K. Rowling: 3 alıntı
      • Jane Austen: 3 alıntı
   📝 En uzun alıntı: 26 kelime
      “There are only two ways to live your life. One is...


---

## 📊 Örnek 5: Pandas ile Veri İşleme

**Data Science ve analiz için en güçlü araç**

### 🎯 Pandas'ın Avantajları:
- **DataFrame** yapısı
- **Güçlü analiz** fonksiyonları
- **Multiple format** desteği
- **Data cleaning** araçları
- **Visualization** entegrasyonu

In [7]:
def process_and_save_with_pandas(data):
    """
    Pandas ile veri işleme ve kaydetme
    """
    
    print(f"📊 PANDAS İLE VERİ İŞLEME")
    print("=" * 40)
    
    if not data:
        print("❌ İşlenecek veri yok!")
        return
    
    try:
        # DataFrame oluştur
        df = pd.DataFrame(data)
        
        print(f"✅ DataFrame oluşturuldu: {df.shape[0]} satır, {df.shape[1]} sütun")
        
        # Temel bilgiler
        print(f"\n📋 DATAFRAME BİLGİLERİ:")
        print(f"   📊 Sütunlar: {list(df.columns)}")
        print(f"   📏 Boyut: {df.shape}")
        print(f"   💾 Hafıza kullanımı: {df.memory_usage(deep=True).sum():,} bytes")
        
        # Veri temizleme
        print(f"\n🧹 VERİ TEMİZLEME:")
        
        # Duplicate kontrolü
        duplicates = df.duplicated(subset=['text']).sum()
        print(f"   📄 Duplicate alıntı: {duplicates}")
        
        if duplicates > 0:
            df_clean = df.drop_duplicates(subset=['text'])
            print(f"   ✅ Duplicates temizlendi: {len(df_clean)} unique alıntı")
        else:
            df_clean = df.copy()
        
        # Missing value kontrolü
        missing_values = df_clean.isnull().sum().sum()
        print(f"   ❓ Eksik değer: {missing_values}")
        
        # Veri dönüştürmeleri
        print(f"\n🔄 VERİ DÖNÜŞTÜRMELERİ:")
        
        # Datetime dönüştürme
        df_clean['scraped_at'] = pd.to_datetime(df_clean['scraped_at'])
        print(f"   📅 Tarih dönüştürüldü: {df_clean['scraped_at'].dtype}")
        
        # Karakter sayısı hesaplama
        df_clean['char_count'] = df_clean['text'].str.len()
        print(f"   🔤 Karakter sayısı eklendi")
        
        # Yazar normalizasyonu
        df_clean['author_normalized'] = df_clean['author'].str.title().str.strip()
        print(f"   ✏️ Yazar isimleri normalize edildi")
        
        # İstatistiksel analiz
        print(f"\n📈 İSTATİSTİKSEL ANALİZ:")
        
        # Kelime sayısı istatistikleri
        word_stats = df_clean['word_count'].describe()
        print(f"   📝 Kelime sayısı istatistikleri:")
        print(f"      Ortalama: {word_stats['mean']:.1f}")
        print(f"      Minimum: {word_stats['min']:.0f}")
        print(f"      Maksimum: {word_stats['max']:.0f}")
        print(f"      Medyan: {word_stats['50%']:.1f}")
        
        # Yazar analizi
        author_counts = df_clean['author_normalized'].value_counts()
        print(f"\n   ✏️ Yazar analizi:")
        print(f"      Toplam yazar: {len(author_counts)}")
        print(f"      En aktif yazar: {author_counts.index[0]} ({author_counts.iloc[0]} alıntı)")
        
        # Tag analizi
        all_tags = []
        for tags in df_clean['tags']:
            all_tags.extend(tags)
        
        tag_series = pd.Series(all_tags)
        top_tags = tag_series.value_counts().head(5)
        print(f"\n   🏷️ En popüler 5 tag:")
        for tag, count in top_tags.items():
            print(f"      • {tag}: {count} kez")
        
        # Klasör oluştur
        output_dir = 'scraped_data'
        os.makedirs(output_dir, exist_ok=True)
        
        # Multiple format kaydetme
        print(f"\n💾 MULTIPLE FORMAT KAYDETME:")
        
        # CSV kaydet
        csv_path = os.path.join(output_dir, 'quotes_pandas.csv')
        df_clean.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"   📄 CSV: {csv_path} ({os.path.getsize(csv_path):,} bytes)")
        
        # Excel kaydet (eğer openpyxl kuruluysa)
        try:
            excel_path = os.path.join(output_dir, 'quotes_pandas.xlsx')
            df_clean.to_excel(excel_path, index=False, engine='openpyxl')
            print(f"   📊 Excel: {excel_path} ({os.path.getsize(excel_path):,} bytes)")
        except ImportError:
            print(f"   ⚠️ Excel kaydetme için: pip install openpyxl")
        
        # Parquet kaydet (eğer pyarrow kuruluysa)
        try:
            parquet_path = os.path.join(output_dir, 'quotes_pandas.parquet')
            df_clean.to_parquet(parquet_path, index=False)
            print(f"   🗜️ Parquet: {parquet_path} ({os.path.getsize(parquet_path):,} bytes)")
        except ImportError:
            print(f"   ⚠️ Parquet kaydetme için: pip install pyarrow")
        
        # JSON kaydet
        json_path = os.path.join(output_dir, 'quotes_pandas.json')
        df_clean.to_json(json_path, orient='records', indent=2, force_ascii=False)
        print(f"   🌐 JSON: {json_path} ({os.path.getsize(json_path):,} bytes)")
        
        # Özet rapor oluştur
        summary_report = {
            'total_quotes': len(df_clean),
            'unique_authors': len(author_counts),
            'total_tags': len(tag_series.unique()),
            'avg_word_count': word_stats['mean'],
            'collection_date': df_clean['scraped_at'].max().isoformat(),
            'top_author': author_counts.index[0],
            'top_tag': top_tags.index[0]
        }
        
        summary_path = os.path.join(output_dir, 'summary_report.json')
        with open(summary_path, 'w', encoding='utf-8') as f:
            json.dump(summary_report, f, indent=2, ensure_ascii=False)
        
        print(f"   📋 Özet rapor: {summary_path}")
        
        return df_clean
        
    except Exception as e:
        print(f"❌ Pandas işleme hatası: {e}")
        return None

# Pandas ile işleme test et
processed_df = process_and_save_with_pandas(sample_data)

if processed_df is not None:
    print(f"\n🎉 Pandas ile veri işleme tamamlandı!")
    print(f"   DataFrame boyutu: {processed_df.shape}")
    print(f"   Sütunlar: {list(processed_df.columns)}")

📊 PANDAS İLE VERİ İŞLEME
✅ DataFrame oluşturuldu: 30 satır, 8 sütun

📋 DATAFRAME BİLGİLERİ:
   📊 Sütunlar: ['id', 'text', 'author', 'tags', 'tags_str', 'word_count', 'scraped_at', 'page_number']
   📏 Boyut: (30, 8)
   💾 Hafıza kullanımı: 21,575 bytes

🧹 VERİ TEMİZLEME:
   📄 Duplicate alıntı: 20
   ✅ Duplicates temizlendi: 10 unique alıntı
   ❓ Eksik değer: 0

🔄 VERİ DÖNÜŞTÜRMELERİ:
   📅 Tarih dönüştürüldü: datetime64[ns]
   🔤 Karakter sayısı eklendi
   ✏️ Yazar isimleri normalize edildi

📈 İSTATİSTİKSEL ANALİZ:
   📝 Kelime sayısı istatistikleri:
      Ortalama: 17.1
      Minimum: 9
      Maksimum: 26
      Medyan: 17.5

   ✏️ Yazar analizi:
      Toplam yazar: 8
      En aktif yazar: Albert Einstein (3 alıntı)

   🏷️ En popüler 5 tag:
      • inspirational: 3 kez
      • humor: 2 kez
      • life: 2 kez
      • change: 1 kez
      • obvious: 1 kez

💾 MULTIPLE FORMAT KAYDETME:
   📄 CSV: scraped_data\quotes_pandas.csv (2,372 bytes)
   ⚠️ Excel kaydetme için: pip install openpyxl
   ⚠️ P

C:\Users\iamem\AppData\Local\Temp\ipykernel_21160\1018590363.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['scraped_at'] = pd.to_datetime(df_clean['scraped_at'])
C:\Users\iamem\AppData\Local\Temp\ipykernel_21160\1018590363.py:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['char_count'] = df_clean['text'].str.len()
C:\Users\iamem\AppData\Local\Temp\ipykernel_21160\1018590363.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try usin

---

## 🔄 Örnek 6: Incremental Updates

**Büyük projeler için: Yeni veriyi mevcut veriye ekleme**

### 🎯 Incremental Update'in Avantajları:
- **Zaman tasarrufu** - Sadece yeni veri
- **Bandwidth tasarrufu** - Az veri transferi
- **Crash recovery** - Kaldığı yerden devam
- **Large datasets** - Büyük veri setleri için ideal

In [8]:
class IncrementalDataSaver:
    """
    Incremental veri kaydetme ve güncelleme sınıfı
    """
    
    def __init__(self, base_dir='scraped_data'):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)
        
        # Dosya yolları
        self.csv_file = os.path.join(base_dir, 'quotes_incremental.csv')
        self.json_file = os.path.join(base_dir, 'quotes_incremental.json')
        self.db_file = os.path.join(base_dir, 'quotes_incremental.db')
        self.state_file = os.path.join(base_dir, 'scraping_state.json')
        
        # State yükle
        self.state = self.load_state()
    
    def load_state(self):
        """Scraping durumunu yükler"""
        if os.path.exists(self.state_file):
            try:
                with open(self.state_file, 'r', encoding='utf-8') as f:
                    state = json.load(f)
                print(f"📊 State yüklendi: {state['total_records']} kayıt mevcut")
                return state
            except Exception as e:
                print(f"⚠️ State yüklenemedi: {e}")
        return {
            'total_records': 0,
            'last_updated': None,
            'last_id': 0,
            'last_page': 0
        }
    
    def save_state(self):
        """Scraping durumunu kaydeder"""
        try:
            with open(self.state_file, 'w', encoding='utf-8') as f:
                json.dump(self.state, f, indent=2, ensure_ascii=False)
            print(f"💾 State kaydedildi: {self.state['total_records']} kayıt")
        except Exception as e:
            print(f"❌ State kaydedilemedi: {e}")
    
    def add_new_data(self, new_data):
        """Yeni veriyi mevcut veriye ekler"""
        print(f"\n🔄 INCREMENTAL DATA UPDATE")
        print(f"   📥 Yeni veri: {len(new_data)} kayıt")
        print(f"   📊 Mevcut veri: {self.state['total_records']} kayıt")
        print("=" * 40)
        if not new_data:
            print("❌ Eklenecek yeni veri yok!")
            return
        try:
            # ID'leri yeniden düzenle
            for i, item in enumerate(new_data):
                item['id'] = self.state['last_id'] + i + 1
            self.append_to_csv(new_data)
            self.update_json(new_data)
            self.update_sqlite(new_data)
            self.state['total_records'] += len(new_data)
            self.state['last_updated'] = datetime.now().isoformat()
            self.state['last_id'] = max(item['id'] for item in new_data)
            self.state['last_page'] = max(item['page_number'] for item in new_data)
            self.save_state()
            print(f"\n✅ Incremental update tamamlandı!")
            print(f"   📈 Yeni toplam: {self.state['total_records']} kayıt")
            print(f"   🆔 Son ID: {self.state['last_id']}")
            print(f"   📄 Son sayfa: {self.state['last_page']}")
        except Exception as e:
            print(f"❌ Incremental update hatası: {e}")
    
    def append_to_csv(self, new_data):
        """CSV dosyasına yeni veri ekler"""
        file_exists = os.path.exists(self.csv_file)
        with open(self.csv_file, 'a', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['id', 'text', 'author', 'tags_str', 'word_count', 'scraped_at', 'page_number']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            for item in new_data:
                csv_row = {field: item.get(field, '') for field in fieldnames}
                writer.writerow(csv_row)
        print(f"   📄 CSV güncellendi: +{len(new_data)} kayıt")
    
    def update_json(self, new_data):
        """JSON dosyasını günceller"""
        existing_data = []
        if os.path.exists(self.json_file):
            with open(self.json_file, 'r', encoding='utf-8') as f:
                existing_data = json.load(f)
        combined_data = existing_data + new_data
        with open(self.json_file, 'w', encoding='utf-8') as f:
            json.dump(combined_data, f, ensure_ascii=False, indent=2)
        print(f"   🌐 JSON güncellendi: {len(existing_data)} + {len(new_data)} = {len(combined_data)}")
    
    def update_sqlite(self, new_data):
        """SQLite database'i günceller"""
        conn = sqlite3.connect(self.db_file)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS quotes (
                id INTEGER PRIMARY KEY,
                text TEXT NOT NULL,
                author TEXT NOT NULL,
                tags TEXT,
                word_count INTEGER,
                scraped_at TIMESTAMP,
                page_number INTEGER
            )
        ''')
        for item in new_data:
            cursor.execute('''
                INSERT OR REPLACE INTO quotes 
                (id, text, author, tags, word_count, scraped_at, page_number)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (
                item['id'],
                item['text'],
                item['author'],
                item['tags_str'],
                item['word_count'],
                item['scraped_at'],
                item['page_number']
            ))
        conn.commit()
        cursor.execute("SELECT COUNT(*) FROM quotes")
        total_records = cursor.fetchone()[0]
        conn.close()
        print(f"   🗃️ SQLite güncellendi: {total_records} toplam kayıt")
    
    def get_stats(self):
        """Mevcut veri istatistiklerini döndürür"""
        return {
            'total_records': self.state['total_records'],
            'last_updated': self.state['last_updated'],
            'last_id': self.state['last_id'],
            'last_page': self.state['last_page'],
            'csv_exists': os.path.exists(self.csv_file),
            'json_exists': os.path.exists(self.json_file),
            'db_exists': os.path.exists(self.db_file)
        }

# Incremental saver test et
incremental_saver = IncrementalDataSaver()
print("🔄 İLK VERİ SETİNİ EKLİYORUZ:")
incremental_saver.add_new_data(sample_data[:5])
print("\n🔄 İKİNCİ VERİ SETİNİ EKLİYORUZ:")
incremental_saver.add_new_data(sample_data[5:])
final_stats = incremental_saver.get_stats()
print(f"\n📊 FINAL İSTATİSTİKLER:")
for key, value in final_stats.items():
    print(f"   {key}: {value}")


🔄 İLK VERİ SETİNİ EKLİYORUZ:

🔄 INCREMENTAL DATA UPDATE
   📥 Yeni veri: 5 kayıt
   📊 Mevcut veri: 0 kayıt
   📄 CSV güncellendi: +5 kayıt
   🌐 JSON güncellendi: 0 + 5 = 5
   🗃️ SQLite güncellendi: 5 toplam kayıt
💾 State kaydedildi: 5 kayıt

✅ Incremental update tamamlandı!
   📈 Yeni toplam: 5 kayıt
   🆔 Son ID: 5
   📄 Son sayfa: 1

🔄 İKİNCİ VERİ SETİNİ EKLİYORUZ:

🔄 INCREMENTAL DATA UPDATE
   📥 Yeni veri: 25 kayıt
   📊 Mevcut veri: 5 kayıt
   📄 CSV güncellendi: +25 kayıt
   🌐 JSON güncellendi: 5 + 25 = 30
   🗃️ SQLite güncellendi: 30 toplam kayıt
💾 State kaydedildi: 30 kayıt

✅ Incremental update tamamlandı!
   📈 Yeni toplam: 30 kayıt
   🆔 Son ID: 30
   📄 Son sayfa: 3

📊 FINAL İSTATİSTİKLER:
   total_records: 30
   last_updated: 2025-08-18T17:53:24.988428
   last_id: 30
   last_page: 3
   csv_exists: True
   json_exists: True
   db_exists: True
